# Lecture 7 self-check: Floating point arithmetic: answers

The answers to the Lecture 7 self-check, with a short explanation for each, including why the tempting wrong options are wrong. You can check most of them by running the code yourself.

### Question 1

What is $(13.375)_{10}$ in binary?

- **(A)** $(1011.110)_2$
- **(B)** $(1101.110)_2$
- **(C)** $(1011.011)_2$
- **(D)** $(1101.011)_2$

**Answer: (D)**

**Integer part.** Divide by 2 and record the remainders: 13 ÷ 2 = 6 remainder **1**, 6 ÷ 2 = 3 remainder **0**, 3 ÷ 2 = 1 remainder **1**, 1 ÷ 2 = 0 remainder **1**. The first remainder is the least significant bit, so read them from the bottom up: $(13)_{10} = (1101)_2$.

**Fractional part.** Multiply by 2 and record the integer part: 0.375 × 2 = **0**.75, 0.75 × 2 = **1**.5, 0.5 × 2 = **1**.0 (stop, the fraction is 0). The first integer part is the first bit after the point, so read them from the top down: $(0.375)_{10} = (0.011)_2$.

Together, $(13.375)_{10} = (1101.011)_2$. Check: $8 + 4 + 1 + \frac{1}{4} + \frac{1}{8} = 13.375$. Reading the remainders from the top gives 1011 (= 11), and reading the fraction's bits from the bottom gives .110 (= 0.75).


### Question 2

What are the 32 bits of $-9.6$ in IEEE-754 single precision? Use round to nearest. Each option shows the sign bit, the 8 exponent bits and the 23 fraction bits, separated by spaces.

- **(A)** `1 10000010 10011001100110011001101`
- **(B)** `1 10000011 00110011001100110011010`
- **(C)** `1 10000011 10011001100110011001101`
- **(D)** `1 10000010 00110011001100110011010`

**Answer: (D)**

1. **Sign:** $x < 0$, so $s = 1$.
2. **Binary:** $9 = (1001)_2$, and $0.6 \times 2 = 1.2$, $0.2 \times 2 = 0.4$, $0.4 \times 2 = 0.8$, $0.8 \times 2 = 1.6$, $0.6 \times 2 = \ldots$, so $0.6 = (0.1001\,1001\ldots)_2$ repeats. $9.6 = (1001.1001\,1001\ldots)_2$.
3. **Normalise:** move the point 3 places to the left: $(1.0011\,0011\,0011\ldots)_2 \times 2^{3}$. The leading 1 is not stored.
4. **Round:** the 23 bits after the point are `00110011001100110011001`, and the next bit is 1 (and the bits after it are not all 0), so round up: $f$ = `00110011001100110011010`.
5. **Exponent:** $e - 127 = 3$, so $e = 130 = (10000010)_2$.

So $-9.6$ is stored as `1 10000010 00110011001100110011010` (the nearest single-precision number, $-9.6000003814\ldots$). Exponent `10000011` (131) comes from taking the 4 integer bits of 1001 as the exponent instead of the 3 places the point moves. Fraction `10011001...` comes from storing the leading 1 in the fraction, which shifts every bit one place. Normalising to $(0.1001\,1001\ldots)_2 \times 2^{4}$ and storing everything after the point gives both of those changes at once.


### Question 3

Which decimal number is stored in these IEEE-754 single-precision bits? They show the sign bit, the 8 exponent bits and the 23 fraction bits, separated by spaces.

`1 10000010 01100000000000000000000`

- **(A)** $-3.0$
- **(B)** $-11.0$
- **(C)** $11.0$
- **(D)** $-5.5$

**Answer: (B)**

- **Sign:** $s = 1$, so $(-1)^{1} = -1$: the number is negative.
- **Exponent:** $e = (10000010)_2 = 130$, and the bias is $2^{7} - 1 = 127$, so the real exponent is $130 - 127 = 3$.
- **Mantissa:** the hidden 1 is added in front of the fraction: $1.f = (1.011)_2 = 1 + \frac{1}{4} + \frac{1}{8} = 1.375$.

$x = -1.375 \times 2^{3} = -11.0$. Leaving out the hidden 1 gives $-0.375 \times 8 = -3.0$; a bias of 128 gives $-1.375 \times 2^{2} = -5.5$; and $s = 1$ means negative, not positive.


### Question 4

What does this print?

```python
big = 1e300 * 1e10
small = 1e-300 * 1e-30
print(big, small, big * small)
```

- **(A)** `inf 0.0 0.0`
- **(B)** Nothing: the first line raises an `OverflowError`
- **(C)** `inf 5e-324 inf`
- **(D)** `inf 0.0 nan`

**Answer: (D)**

$10^{310}$ is larger than the largest double (about $1.8 \times 10^{308}$): the product overflows, and IEEE-754 gives `inf` without an error (only some Python operations, such as `2.0**10000`, raise an `OverflowError`). $10^{-330}$ is smaller than even the smallest subnormal double ($\approx 4.9 \times 10^{-324}$): it underflows to `0.0`, rather than being kept at the smallest number. Finally $\infty \times 0$ has no meaningful value, so it is `nan` (not a number), not `0.0`.


### Question 5

Machine epsilon for double precision (`np.finfo(float).eps`) is `2.220446049250313e-16`. What does this print?

```python
x = 1.0 + 1.5e-16
print(x == 1.0, x - 1.0)
```

- **(A)** `True 0.0`
- **(B)** `False 2.220446049250313e-16`
- **(C)** `False 1.5e-16`

**Answer: (B)**

The doubles next to 1 are $1$ and $1 + \epsilon$, with nothing in between. The exact sum $1 + 1.5 \times 10^{-16}$ lies between them, and it is rounded to the nearer one. Halfway is $1 + \epsilon/2 \approx 1 + 1.11 \times 10^{-16}$, and $1.5 \times 10^{-16}$ is above that, so `x` becomes $1 + \epsilon$: `x == 1.0` is `False` and `x - 1.0` is exactly $\epsilon$. Below $\epsilon/2$ (e.g. `1.0 + 1e-16`), `x` would round back to `1.0`. The sum is never exactly $1 + 1.5 \times 10^{-16}$, because that is not a machine number.


### Question 6

What does this print?

```python
first = 0.1 + 0.7 == 0.8
second = 0.5 + 0.25 == 0.75
print(first, second)
```

- **(A)** `False False`
- **(B)** `True True`
- **(C)** `False True`

**Answer: (C)**

0.1, 0.7 and 0.8 have infinite binary expansions (like 1/3 in decimal), so each is stored rounded, and `0.1 + 0.7` is `0.7999999999999999`, not the stored 0.8. But 0.5, 0.25 and 0.75 are $2^{-1}$, $2^{-2}$ and $2^{-1} + 2^{-2}$: they are machine numbers, stored exactly, and their sum is exact too. Not every float is inexact, but you can't tell by looking at the decimal, which is why floats are compared with a tolerance.


### Question 7

Machine epsilon for double precision (`np.finfo(float).eps`) is `2.220446049250313e-16`. What does this print?

```python
a = 1e16
b = (a + 1.5) - a
print(b)
```

- **(A)** `0.0`
- **(B)** `1.5`
- **(C)** `2.0`

**Answer: (C)**

Machine numbers are not equally spaced: the gap between neighbouring doubles is about $\epsilon |x|$. Near $10^{16}$ that is about $2.2 \times 10^{-16} \times 10^{16} \approx 2$ (exactly 2, as `math.ulp(1e16)` shows), so the doubles near `a` are `a`, `a + 2`, `a + 4`, .... The exact sum `a + 1.5` is not a machine number, and it is rounded to the nearest one, `a + 2`. Subtracting `a` is then exact, so `b` is `2.0`. Getting `1.5` assumes the doubles are $\epsilon$ apart everywhere; getting `0.0` assumes the extra bits are chopped off instead of rounded to the nearest machine number.


### Question 8

Single precision (`np.float32`) has machine epsilon of about $1.2 \times 10^{-7}$. What does this print?

```python
import numpy as np

big = np.float32(1.0)
smalls = [np.float32(1e-8)] * 1000

a = big
for t in smalls:
    a += t

b = np.float32(0.0)
for t in smalls:
    b += t
b += big
print(a, b)
```

- **(A)** `1.00001 1.0`
- **(B)** `1.00001 1.00001`
- **(C)** `1.0 1.00001`

**Answer: (C)**

In `a`, each $10^{-8}$ is added to a sum of about 1. That is less than half the gap between single-precision numbers near 1 ($\epsilon/2 \approx 6 \times 10^{-8}$), so every addition rounds back to 1.0: all 1000 terms are lost. In `b`, the small terms are added to each other first, building up to $10^{-5}$, which is large enough to survive being added to 1.0: `b` is 1.00001. Floating-point addition is not associative, so the order matters, and adding the small numbers first is the more accurate order.


### Question 9

What does this print?

```python
import math

x = 1e16
naive = math.sqrt(x + 1) - math.sqrt(x)
better = 1 / (math.sqrt(x + 1) + math.sqrt(x))
print(naive, better)
```

- **(A)** `0.0 5e-09`
- **(B)** `0.0 0.0`
- **(C)** `5e-09 5e-09`

**Answer: (A)**

Near $10^{16}$ the doubles are 2 apart, so `x + 1` rounds back to `x`, and `naive` subtracts two equal numbers: `0.0`, with no correct digits at all. `better` is the same expression multiplied and divided by the conjugate: $\sqrt{x+1} - \sqrt{x} = 1/(\sqrt{x+1} + \sqrt{x})$. It only adds two positive numbers and divides, so it is accurate: $1/(10^{8} + 10^{8}) = 5 \times 10^{-9}$. Expressions that are equal in exact arithmetic can give very different floating-point results.


### Question 10

The largest double is about $1.8 \times 10^{308}$. What does this print?

```python
a = 1e308
b = 1e308
first = (a + b) / 2
second = a / 2 + b / 2
print(first, second)
```

- **(A)** `inf 1e+308`
- **(B)** Nothing: the line `first = (a + b) / 2` raises an `OverflowError`
- **(C)** `1e+308 1e+308`

**Answer: (A)**

The average of `a` and `b` is $10^{308}$, which is representable, but `(a + b) / 2` first computes `a + b` $= 2 \times 10^{308}$, which overflows to `inf` (without an error), and `inf / 2` is still `inf`. `a / 2 + b / 2` keeps every intermediate value moderate, $5 \times 10^{307}$, and gives `1e+308`. A representable result does not mean the intermediate values are.


### Question 11

What does this print?

```python
def count_steps(step):
    x = 0.0
    n = 0
    while x != 1.0 and n < 100:
        x += step
        n += 1
    return n


first = count_steps(0.25)
second = count_steps(0.1)
print(first, second)
```

- **(A)** `100 100`
- **(B)** `4 100`
- **(C)** `4 10`

**Answer: (B)**

0.25 is $2^{-2}$, a machine number, and 0.25, 0.5, 0.75 and 1.0 are all exact, so with `step = 0.25` `x` becomes exactly `1.0` after 4 steps and the loop stops. 0.1 is not a machine number: after 10 steps `x` is `0.9999999999999999`, never exactly `1.0`, so `x != 1.0` stays true and only the limit `n < 100` stops the loop. Without that limit the loop would never end. Use a tolerance (e.g. `math.isclose`) and always a maximum number of iterations.


### Question 12

What does this print?

```python
import math

import numpy as np

first = math.isclose(1e-12, 3e-12)
second = np.isclose(1e-12, 3e-12)
print(first, second)
```

- **(A)** `False True`
- **(B)** `False False`
- **(C)** `True False`
- **(D)** `True True`

**Answer: (A)**

`math.isclose(a, b)` uses only a relative tolerance by default (`rel_tol=1e-09`, `abs_tol=0.0`): it checks `abs(a - b) <= 1e-09 * max(abs(a), abs(b))`. Here `abs(a - b)` is `2e-12`, a relative difference of about 0.67, far above `1e-09`, so `False`. `np.isclose(a, b)` checks `abs(a - b) <= atol + rtol * abs(b)`, with `atol=1e-08` and `rtol=1e-05` by default. `2e-12` is far below `atol`, so `True`: any two numbers smaller than about `1e-08` compare as close. For tiny numbers, pass `atol=0` to `np.isclose`.
